In [1]:
# --- Cell 0: Setup (portable) & .bin reader ---
import os, glob, numpy as np

# Root = current working directory (portable)
ROOT_DIR = os.getcwd()
FOREHEAD_DIR = os.path.join(ROOT_DIR, "Forehead")
SCLERA_DIR   = os.path.join(ROOT_DIR, "Sclera")

# Task spec (width, height, channels)
SPEC = {
    "forehead": {"W":128, "H":128, "C":3},
    "sclera"  : {"W":256, "H": 64, "C":3},   # width=256, height=64
}
FPS = 60.0  # per task

def read_bin_rgb(path, region):
    """Decode .bin into numpy array (N, H, W, 3)."""
    W, H, C = SPEC[region]["W"], SPEC[region]["H"], SPEC[region]["C"]
    raw = np.fromfile(path, dtype=np.uint8)
    frame_bytes = W * H * C
    if raw.size % frame_bytes != 0:
        raise ValueError(f"{os.path.basename(path)}: size {raw.size} not divisible by {frame_bytes} (W={W},H={H},C={C})")
    N = raw.size // frame_bytes
    arr = raw.reshape(N, H, W, C)
    return arr, N

# Collect files
forehead_files = sorted(glob.glob(os.path.join(FOREHEAD_DIR, "*.bin")))
sclera_files   = sorted(glob.glob(os.path.join(SCLERA_DIR, "*.bin")))
print(f"Found {len(forehead_files)} forehead bins, {len(sclera_files)} sclera bins")
if forehead_files:
    arr, N = read_bin_rgb(forehead_files[0], "forehead")
    print(f"Sample forehead: {os.path.basename(forehead_files[0])} -> {arr.shape}, N={N}, FPS={FPS}")
if sclera_files:
    arr, N = read_bin_rgb(sclera_files[0], "sclera")
    print(f"Sample sclera  : {os.path.basename(sclera_files[0])} -> {arr.shape}, N={N}, FPS={FPS}")


Found 5 forehead bins, 5 sclera bins
Sample forehead: forehead_all_128_128_1.bin -> (13564, 128, 128, 3), N=13564, FPS=60.0
Sample sclera  : sclera-eye_left_all_256_64_1.bin -> (13564, 64, 256, 3), N=13564, FPS=60.0


In [2]:
# --- Cell 1: Core helpers (extraction, normalization, filtering, HR from Welch) ---
import numpy as np
from scipy.signal import butter, filtfilt, welch, detrend

# Color-channel baseline
def mean_green(frames):
    """(N,H,W,3) -> (N,) mean of Green channel per frame."""
    return frames[..., 1].mean(axis=(1,2)).astype(np.float32)

# Normalization & filtering
def detrend_linear(x):
    return detrend(x, type="linear")

def zscore(x):
    x = np.asarray(x, dtype=np.float64)
    mu, sd = np.mean(x), np.std(x) + 1e-8
    return (x - mu) / sd

# Assignment band
F_LO, F_HI = 0.7, 4.0   # Hz  (≈ 42–240 BPM)

def butter_bandpass(lo, hi, fs, order=3):
    nyq = 0.5 * fs
    return butter(order, [lo/nyq, hi/nyq], btype="band")

def bandpass_filter(x, fs, lo=F_LO, hi=F_HI, order=3):
    b, a = butter_bandpass(lo, hi, fs, order)
    return filtfilt(b, a, x)

# Stable Welch peak picker (prevents numeric blowups)
def welch_hr_stable(sig, fs, bpm_min=40, bpm_max=240, nperseg=4096, noverlap=2048):
    """
    Welch PSD -> dominant BPM with SAFE parabolic interpolation.
    Keeps assignment band 0.7–4.0 Hz (~42–240 BPM); we clip to [40,240] for rounding.
    """
    sig = np.asarray(sig, dtype=np.float64)
    nperseg = min(int(nperseg), len(sig))
    noverlap = min(int(noverlap), max(0, nperseg//2))

    f, Pxx = welch(sig, fs=fs, nperseg=nperseg, noverlap=noverlap)
    f = f.astype(np.float64); Pxx = Pxx.astype(np.float64)

    bpm_axis = f * 60.0
    mask = (bpm_axis >= bpm_min) & (bpm_axis <= bpm_max)
    if not np.any(mask):
        return np.nan, f, Pxx

    idx_mask = np.where(mask)[0]
    i = idx_mask[np.argmax(Pxx[mask])]

    # Safe parabolic interpolation around i
    fpk = f[i]
    if 1 <= i < len(Pxx)-1:
        y0, y1, y2 = Pxx[i-1], Pxx[i], Pxx[i+1]
        denom = (y0 - 2.0*y1 + y2)
        # require strict local max and concavity
        if (y1 > y0) and (y1 > y2) and (denom < -1e-18):
            delta = 0.5 * (y0 - y2) / denom          # sub-bin shift
            delta = float(np.clip(delta, -0.5, 0.5)) # clamp within half-bin
            fpk = f[i] + delta * (f[1] - f[0])

    bpm = float(max(0.0, fpk * 60.0))
    return bpm, f, Pxx


In [3]:
# --- Cell 2: POS (Plane-Orthogonal-to-Skin), strict implementation ---
import numpy as np

def roi_mean_rgb(frames):
    """(N,H,W,3) -> (N,3) float32, spatial mean RGB per frame."""
    return frames.mean(axis=(1,2)).astype(np.float32)

def pos_from_rgb_strict(rgb, fps, win_sec=1.6, overlap=0.5, eps=1e-8):
    """
    POS exactly as in the paper:
      1) window-wise normalization: Cn = RGB/mean(RGB_window) - 1
      2) S1 = G - B,  S2 = G + B - 2R
      3) h = S1 + (std(S1)/std(S2))*S2   (ADDITION)
      4) overlap-add to full-length signal
    """
    N = rgb.shape[0]
    L = max(2, int(round(win_sec * fps)))
    step = max(1, int(round(L * (1.0 - overlap))))

    out = np.zeros(N, dtype=np.float32)
    wsum = np.zeros(N, dtype=np.float32)

    i = 0
    while i < N:
        j = min(i + L, N)
        seg = rgb[i:j].copy()
        mu = seg.mean(axis=0)
        Cn = seg / (mu + eps) - 1.0     # (wlen,3)

        Rn, Gn, Bn = Cn[:,0], Cn[:,1], Cn[:,2]
        S1 = Gn - Bn
        S2 = Gn + Bn - 2.0 * Rn

        s1 = S1 - S1.mean()
        s2 = S2 - S2.mean()
        alpha = (np.std(s1) + eps) / (np.std(s2) + eps)
        h = s1 + alpha * s2
        h = h - h.mean()

        out[i:j] += h
        wsum[i:j] += 1.0

        if j == N: break
        i += step

    wsum[wsum < 1e-8] = 1.0
    return (out / wsum).astype(np.float32)

def pos_from_frames_strict(frames, fps, win_sec=1.6, overlap=0.5):
    rgb = roi_mean_rgb(frames)
    return pos_from_rgb_strict(rgb, fps=fps, win_sec=win_sec, overlap=overlap)


In [4]:
# --- Cell 3: CHROM (de Haan & Jeanne, 2013), strict implementation ---
import numpy as np

def chrom_from_rgb_strict(rgb, fps, win_sec=1.6, overlap=0.5, eps=1e-8):
    """
    CHROM:
      1) window-wise normalization: Cn = RGB/mean(RGB_window) - 1
      2) X = 3*R_n - 2*G_n
         Y = 1.5*R_n + 1.0*G_n - 1.5*B_n
      3) h = X - (std(X)/std(Y)) * Y    (SUBTRACTION)
      4) overlap-add to full-length signal
    """
    N = rgb.shape[0]
    L = max(2, int(round(win_sec * fps)))
    step = max(1, int(round(L * (1.0 - overlap))))

    out = np.zeros(N, dtype=np.float32)
    wsum = np.zeros(N, dtype=np.float32)

    i = 0
    while i < N:
        j = min(i + L, N)
        seg = rgb[i:j].copy()
        mu = seg.mean(axis=0)
        Cn = seg / (mu + eps) - 1.0   # (wlen,3)

        Rn, Gn, Bn = Cn[:,0], Cn[:,1], Cn[:,2]
        X = 3.0*Rn - 2.0*Gn
        Y = 1.5*Rn + 1.0*Gn - 1.5*Bn

        x = X - X.mean()
        y = Y - Y.mean()
        alpha = (np.std(x) + eps) / (np.std(y) + eps)
        h = x - alpha * y
        h = h - h.mean()

        out[i:j] += h
        wsum[i:j] += 1.0

        if j == N: break
        i += step

    wsum[wsum < 1e-8] = 1.0
    return (out / wsum).astype(np.float32)

def chrom_from_frames_strict(frames, fps, win_sec=1.6, overlap=0.5):
    rgb = roi_mean_rgb(frames)  # uses helper from POS cell
    return chrom_from_rgb_strict(rgb, fps=fps, win_sec=win_sec, overlap=overlap)


In [5]:
# --- Cell 4: Batch process (Green vs POS vs CHROM) & print/save summary ---
import os, re, numpy as np, pandas as pd

def patient_id_from_path(p):
    """Extract trailing number as patient id; fallback to stem."""
    base = os.path.splitext(os.path.basename(p))[0]
    m = re.search(r"(\d+)$", base)
    return m.group(1) if m else base

# Pair forehead <-> sclera
fh_map = {patient_id_from_path(p): p for p in forehead_files}
pairs = []
for p in sclera_files:
    pid = patient_id_from_path(p)
    if pid in fh_map:
        pairs.append((pid, fh_map[pid], p))
print(f"Paired {len(pairs)} patient(s).")

def bpm_from_signal(sig):
    sig = zscore(detrend_linear(sig))
    sig_bp = bandpass_filter(sig, fs=FPS)  # 0.7–4.0 Hz per task
    bpm, _, _ = welch_hr_stable(sig_bp, fs=FPS, bpm_min=40, bpm_max=240, nperseg=4096, noverlap=2048)
    return bpm

def process_region_all(frames):
    # GREEN
    g = mean_green(frames)
    bpm_g = bpm_from_signal(g)

    # POS
    pos = pos_from_frames_strict(frames, fps=FPS, win_sec=1.6, overlap=0.5)
    bpm_pos = bpm_from_signal(pos)

    # CHROM
    chrom = chrom_from_frames_strict(frames, fps=FPS, win_sec=1.6, overlap=0.5)
    bpm_chrom = bpm_from_signal(chrom)

    return bpm_g, bpm_pos, bpm_chrom

rows = []
for pid, fh_path, sc_path in pairs:
    fh_frames, _ = read_bin_rgb(fh_path, "forehead")
    sc_frames, _ = read_bin_rgb(sc_path, "sclera")

    # Forehead
    fg, fp, fc = process_region_all(fh_frames)
    # Sclera
    sg, sp, sc = process_region_all(sc_frames)

    rows.append({
        "id": pid,
        "bpm_forehead_green": fg,
        "bpm_forehead_pos": fp,
        "bpm_forehead_chrom": fc,
        "bpm_sclera_green": sg,
        "bpm_sclera_pos": sp,
        "bpm_sclera_chrom": sc,
    })

df3 = pd.DataFrame(rows).sort_values("id")
display(df3)

# Pretty print
print("\n=== Heart Rate Estimates (BPM) — GREEN vs POS vs CHROM ===")
for _, r in df3.iterrows():
    print(f"Patient {r['id']}: "
          f"Forehead G/POS/CHROM = {r['bpm_forehead_green']:.2f} / {r['bpm_forehead_pos']:.2f} / {r['bpm_forehead_chrom']:.2f} | "
          f"Sclera   G/POS/CHROM = {r['bpm_sclera_green']:.2f} / {r['bpm_sclera_pos']:.2f} / {r['bpm_sclera_chrom']:.2f}")

# Save CSV
out_csv = "pulse_summary_green_pos_chrom.csv"
df3.to_csv(out_csv, index=False)
print("Saved:", out_csv)


Paired 5 patient(s).


,id,bpm_forehead_green,bpm_forehead_pos,bpm_forehead_chrom,bpm_sclera_green,bpm_sclera_pos,bpm_sclera_chrom
0,1,46.467262,147.888211,55.460130,52.591684,71.453341,57.044443
1,10,58.122751,201.300259,72.110251,51.978809,63.236218,53.708348
2,29,65.258709,77.740815,122.468090,65.938259,86.838491,65.764639
3,5,47.896559,79.855973,79.859341,70.170555,59.222160,59.125214
4,6,61.976514,72.001176,71.941785,73.938353,58.082412,57.997384



=== Heart Rate Estimates (BPM) — GREEN vs POS vs CHROM ===
Patient 1: Forehead G/POS/CHROM = 46.47 / 147.89 / 55.46 | Sclera   G/POS/CHROM = 52.59 / 71.45 / 57.04
Patient 10: Forehead G/POS/CHROM = 58.12 / 201.30 / 72.11 | Sclera   G/POS/CHROM = 51.98 / 63.24 / 53.71
Patient 29: Forehead G/POS/CHROM = 65.26 / 77.74 / 122.47 | Sclera   G/POS/CHROM = 65.94 / 86.84 / 65.76
Patient 5: Forehead G/POS/CHROM = 47.90 / 79.86 / 79.86 | Sclera   G/POS/CHROM = 70.17 / 59.22 / 59.13
Patient 6: Forehead G/POS/CHROM = 61.98 / 72.00 / 71.94 | Sclera   G/POS/CHROM = 73.94 / 58.08 / 58.00
Saved: pulse_summary_green_pos_chrom.csv


In [6]:
# --- Cell 4: Pair forehead<->sclera by patient id, process all ---
import os, re, pandas as pd, numpy as np

def patient_id_from_path(p):
    """Extract trailing number as patient id; fallback to stem."""
    base = os.path.splitext(os.path.basename(p))[0]
    m = re.search(r"(\d+)$", base)
    return m.group(1) if m else base

# Build pairs
fh_map = {patient_id_from_path(p): p for p in forehead_files}
pairs = []
for p in sclera_files:
    pid = patient_id_from_path(p)
    if pid in fh_map:
        pairs.append((pid, fh_map[pid], p))
print(f"Paired {len(pairs)} patient(s).")

def process_region(frames):
    # GREEN
    g = zscore(detrend_linear(mean_green(frames)))
    g_bp = bandpass_filter(g, fs=FPS)
    bpm_g, f_g, Pxx_g = welch_hr_stable(g_bp, fs=FPS, bpm_min=40, bpm_max=240, nperseg=4096, noverlap=2048)

    # POS
    pos = pos_from_frames_strict(frames, fps=FPS, win_sec=1.6, overlap=0.5)
    pos = zscore(detrend_linear(pos))
    pos_bp = bandpass_filter(pos, fs=FPS)
    bpm_pos, f_pos, Pxx_pos = welch_hr_stable(pos_bp, fs=FPS, bpm_min=40, bpm_max=240, nperseg=4096, noverlap=2048)

    return bpm_g, bpm_pos

rows = []
for pid, fh_path, sc_path in pairs:
    fh_frames, _ = read_bin_rgb(fh_path, "forehead")
    sc_frames, _ = read_bin_rgb(sc_path, "sclera")

    bpm_fg, bpm_fp = process_region(fh_frames)  # forehead: green, pos
    bpm_sg, bpm_sp = process_region(sc_frames)  # sclera:   green, pos

    rows.append({
        "id": pid,
        "bpm_forehead_green": bpm_fg,
        "bpm_forehead_pos": bpm_fp,
        "bpm_sclera_green": bpm_sg,
        "bpm_sclera_pos": bpm_sp,
    })

df = pd.DataFrame(rows).sort_values("id")
display(df)


Paired 5 patient(s).


,id,bpm_forehead_green,bpm_forehead_pos,bpm_sclera_green,bpm_sclera_pos
0,1,46.467262,147.888211,52.591684,71.453341
1,10,58.122751,201.300259,51.978809,63.236218
2,29,65.258709,77.740815,65.938259,86.838491
3,5,47.896559,79.855973,70.170555,59.222160
4,6,61.976514,72.001176,73.938353,58.082412


In [10]:
# --- 15-second windowed HR estimation (fixed-middle or best-SNR) ---
import numpy as np
import pandas as pd
import os, re

# ---- spectral SNR helper (within the assignment band 0.7–4.0 Hz) ----
F_LO, F_HI = 0.7, 4.0  # Hz

def spectral_snr(freqs, psd, f_peak, half_width_hz=0.33):
    """
    SNR (dB) = power around f_peak (±half_width_hz) / power in the rest of [F_LO, F_HI].
    """
    if np.isnan(f_peak):
        return np.nan
    band_mask  = (freqs >= F_LO) & (freqs <= F_HI)
    if not np.any(band_mask):
        return np.nan
    peak_mask  = (freqs >= f_peak - half_width_hz) & (freqs <= f_peak + half_width_hz)
    peak_mask &= band_mask
    noise_mask = band_mask & (~peak_mask)
    p_peak  = np.trapz(np.maximum(psd[peak_mask], 0), freqs[peak_mask]) + 1e-12
    p_noise = np.trapz(np.maximum(psd[noise_mask], 0), freqs[noise_mask]) + 1e-12
    return float(10.0*np.log10(p_peak/p_noise))

# ---- window utilities ----
def middle_window(sig, fs, win_sec=15.0):
    N = len(sig)
    L = int(win_sec * fs)
    if L >= N:
        return sig  # whole signal if too short
    start = (N - L) // 2
    return sig[start:start+L]

def best_snr_window(sig, fs, win_sec=15.0, hop_sec=15.0):
    """
    Slide a 15s window with hop, pick the one with the highest spectral SNR (using the assignment band).
    Returns the best windowed signal and its (bpm, snr).
    """
    L   = int(win_sec * fs)
    hop = max(1, int(hop_sec * fs))
    N   = len(sig)
    if L >= N:  # too short; just evaluate whole signal
        sig_win = sig
        sig_win_bp = bandpass_filter(zscore(detrend_linear(sig_win)), fs=fs)
        bpm, freqs, psd = welch_hr_stable(sig_win_bp, fs=fs, bpm_min=40, bpm_max=240,
                                          nperseg=min(4096, len(sig_win_bp)), noverlap=min(2048, len(sig_win_bp)//2))
        snr = spectral_snr(freqs, psd, bpm/60.0 if not np.isnan(bpm) else np.nan)
        return sig_win, bpm, snr

    best = None
    best_tuple = (None, -np.inf)  # (index, snr)
    i = 0
    while i + L <= N:
        seg = sig[i:i+L]
        seg_bp = bandpass_filter(zscore(detrend_linear(seg)), fs=fs)
        bpm, freqs, psd = welch_hr_stable(seg_bp, fs=fs, bpm_min=40, bpm_max=240,
                                          nperseg=min(4096, len(seg_bp)), noverlap=min(2048, len(seg_bp)//2))
        snr = spectral_snr(freqs, psd, bpm/60.0 if not np.isnan(bpm) else np.nan)
        if snr > best_tuple[1]:
            best_tuple = (i, snr)
            best = (seg, bpm, snr)
        i += hop

    # return best
    if best is None:
        # fallback: middle window
        seg = middle_window(sig, fs, win_sec)
        seg_bp = bandpass_filter(zscore(detrend_linear(seg)), fs=fs)
        bpm, freqs, psd = welch_hr_stable(seg_bp, fs=fs, bpm_min=40, bpm_max=240,
                                          nperseg=min(4096, len(seg_bp)), noverlap=min(2048, len(seg_bp)//2))
        snr = spectral_snr(freqs, psd, bpm/60.0 if not np.isnan(bpm) else np.nan)
        return seg, bpm, snr
    return best  # (seg, bpm, snr)

# ---- per-method signal constructors ----
def sig_green(frames):
    return mean_green(frames)

def sig_pos(frames):
    return pos_from_frames_strict(frames, fps=FPS, win_sec=1.6, overlap=0.5)

def sig_chrom(frames):
    return chrom_from_frames_strict(frames, fps=FPS, win_sec=1.6, overlap=0.5)

METHODS = {
    "green": sig_green,
    "pos":   sig_pos,
    "chrom": sig_chrom,
}

# ---- patient pairing (same as before) ----
def patient_id_from_path(p):
    base = os.path.splitext(os.path.basename(p))[0]
    m = re.search(r"(\d+)$", base)
    return m.group(1) if m else base

fh_map = {patient_id_from_path(p): p for p in forehead_files}
pairs = []
for p in sclera_files:
    pid = patient_id_from_path(p)
    if pid in fh_map:
        pairs.append((pid, fh_map[pid], p))
print(f"Paired {len(pairs)} patient(s).")

# ---- run batch with 15s windows ----
def run_windowed(frames, method="green", strategy="best_snr", win_sec=30.0, hop_sec=15.0):
    """
    method: 'green' | 'pos' | 'chrom'
    strategy: 'middle' (fixed middle 15s) | 'best_snr' (scan and pick highest SNR window)
    Returns (bpm, snr)
    """
    sig = METHODS[method](frames)

    if strategy == "middle":
        seg = middle_window(sig, FPS, win_sec)
        seg_bp = bandpass_filter(zscore(detrend_linear(seg)), fs=FPS)
        bpm, freqs, psd = welch_hr_stable(seg_bp, fs=FPS, bpm_min=40, bpm_max=240,
                                          nperseg=min(4096, len(seg_bp)), noverlap=min(2048, len(seg_bp)//2))
        snr = spectral_snr(freqs, psd, bpm/60.0 if not np.isnan(bpm) else np.nan)
        return bpm, snr

    elif strategy == "best_snr":
        seg, bpm, snr = best_snr_window(sig, fs=FPS, win_sec=win_sec, hop_sec=hop_sec)
        return bpm, snr

    else:
        raise ValueError("strategy must be 'middle' or 'best_snr'")

# ---- process all patients (both regions, 3 methods) ----
def process_all_pairs(strategy="best_snr", win_sec=30.0, hop_sec=15.0):
    rows = []
    for pid, fh_path, sc_path in pairs:
        fh_frames, _ = read_bin_rgb(fh_path, "forehead")
        sc_frames, _ = read_bin_rgb(sc_path, "sclera")

        # Forehead
        fg_bpm, fg_snr = run_windowed(fh_frames, "green", strategy, win_sec, hop_sec)
        fp_bpm, fp_snr = run_windowed(fh_frames, "pos"  , strategy, win_sec, hop_sec)
        fc_bpm, fc_snr = run_windowed(fh_frames, "chrom", strategy, win_sec, hop_sec)

        # Sclera
        sg_bpm, sg_snr = run_windowed(sc_frames, "green", strategy, win_sec, hop_sec)
        sp_bpm, sp_snr = run_windowed(sc_frames, "pos"  , strategy, win_sec, hop_sec)
        sc_bpm, sc_snr = run_windowed(sc_frames, "chrom", strategy, win_sec, hop_sec)

        rows.append({
            "id": pid,
            "bpm_forehead_green": fg_bpm, "snr_forehead_green": fg_snr,
            "bpm_forehead_pos":   fp_bpm, "snr_forehead_pos":   fp_snr,
            "bpm_forehead_chrom": fc_bpm, "snr_forehead_chrom": fc_snr,
            "bpm_sclera_green":   sg_bpm, "snr_sclera_green":   sg_snr,
            "bpm_sclera_pos":     sp_bpm, "snr_sclera_pos":     sp_snr,
            "bpm_sclera_chrom":   sc_bpm, "snr_sclera_chrom":   sc_snr,
            "window_sec": win_sec, "hop_sec": hop_sec, "strategy": strategy
        })
    return pd.DataFrame(rows).sort_values("id")

# --- choose strategy here ---
# strategy = "middle"     # fixed middle 15 s
strategy = "best_snr"     # scan 15 s windows, pick best SNR

df_win = process_all_pairs(strategy=strategy, win_sec=30.0, hop_sec=15.0)
display(df_win)

# Pretty print
print(f"\n=== 15s-window HR (strategy={strategy}) — GREEN / POS / CHROM ===")
for _, r in df_win.iterrows():
    print(f"Patient {r['id']}: "
          f"Forehead G/POS/CHROM = {r['bpm_forehead_green']:.2f}/{r['bpm_forehead_pos']:.2f}/{r['bpm_forehead_chrom']:.2f} | "
          f"Sclera   G/POS/CHROM = {r['bpm_sclera_green']:.2f}/{r['bpm_sclera_pos']:.2f}/{r['bpm_sclera_chrom']:.2f} "
          f"(SNR dB FH G/P/C = {r['snr_forehead_green']:.1f}/{r['snr_forehead_pos']:.1f}/{r['snr_forehead_chrom']:.1f}; "
          f"SC G/P/C = {r['snr_sclera_green']:.1f}/{r['snr_sclera_pos']:.1f}/{r['snr_sclera_chrom']:.1f})")

# Save CSV
out_csv = f"pulse_summary_window15_{strategy}.csv"
df_win.to_csv(out_csv, index=False)
print("Saved:", out_csv)


Paired 5 patient(s).


,id,bpm_forehead_green,snr_forehead_green,bpm_forehead_pos,snr_forehead_pos,bpm_forehead_chrom,snr_forehead_chrom,bpm_sclera_green,snr_sclera_green,bpm_sclera_pos,snr_sclera_pos,bpm_sclera_chrom,snr_sclera_chrom,window_sec,hop_sec,strategy
0,1,57.741426,-0.130600,64.190284,-3.618873,55.523025,-3.333667,58.617320,-1.027181,88.146760,-1.326759,88.261965,-1.277981,30.0,15.0,best_snr
1,10,56.407838,-1.415967,59.985681,-3.154300,57.895847,-1.128447,57.737294,1.844391,57.720124,-0.351966,60.229898,-2.226120,30.0,15.0,best_snr
2,29,92.724291,-3.787966,77.838245,-1.787393,123.265513,-3.364274,58.653359,0.605278,78.922876,-1.037390,69.475048,-0.960907,30.0,15.0,best_snr
3,5,47.416755,-1.649084,78.840359,0.174127,64.110144,-1.692097,52.031674,-0.076068,59.954197,0.411579,59.833038,-0.730640,30.0,15.0,best_snr
4,6,61.557328,-1.374025,61.491769,-1.369920,108.062940,-1.379850,73.218135,-0.228621,58.201709,0.115813,55.970443,0.873411,30.0,15.0,best_snr



=== 15s-window HR (strategy=best_snr) — GREEN / POS / CHROM ===
Patient 1: Forehead G/POS/CHROM = 57.74/64.19/55.52 | Sclera   G/POS/CHROM = 58.62/88.15/88.26 (SNR dB FH G/P/C = -0.1/-3.6/-3.3; SC G/P/C = -1.0/-1.3/-1.3)
Patient 10: Forehead G/POS/CHROM = 56.41/59.99/57.90 | Sclera   G/POS/CHROM = 57.74/57.72/60.23 (SNR dB FH G/P/C = -1.4/-3.2/-1.1; SC G/P/C = 1.8/-0.4/-2.2)
Patient 29: Forehead G/POS/CHROM = 92.72/77.84/123.27 | Sclera   G/POS/CHROM = 58.65/78.92/69.48 (SNR dB FH G/P/C = -3.8/-1.8/-3.4; SC G/P/C = 0.6/-1.0/-1.0)
Patient 5: Forehead G/POS/CHROM = 47.42/78.84/64.11 | Sclera   G/POS/CHROM = 52.03/59.95/59.83 (SNR dB FH G/P/C = -1.6/0.2/-1.7; SC G/P/C = -0.1/0.4/-0.7)
Patient 6: Forehead G/POS/CHROM = 61.56/61.49/108.06 | Sclera   G/POS/CHROM = 73.22/58.20/55.97 (SNR dB FH G/P/C = -1.4/-1.4/-1.4; SC G/P/C = -0.2/0.1/0.9)
Saved: pulse_summary_window15_best_snr.csv
